## Whole-slide image inference example

This notebook illustrates how to encode tiles from a whole-slide image using a Triton inference server.

- See https://github.com/PathologyDataScience/simple_triton for details on launching the triton server container
- Run this notebook in a container with `--network=host` so that it can reach the Triton container
- Mount your model repository directory to the triton container

If running this notebook on the same machine as the Triton server, prevent TensorFlow from allocating GPU resources.

In [11]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
import tensorflow as tf

assert len(tf.config.list_physical_devices("GPU")) == 0

## Download sample data

Download the hosted whole-slide image and mask.

In [12]:
import pooch

# download whole slide image and corresponding mask
wsi_path = pooch.retrieve(
    fname="TCGA-AN-A0G0-01Z-00-DX1.svs",
    url="https://drive.usercontent.google.com/download?id=19agE_0cWY582szhOVxp9h3kozRfB4CvV&export=download&confirm=t",
    known_hash="d046f952759ff6987374786768fc588740eef1e54e4e295a684f3bd356c8528f",
    path=str(pooch.os_cache("pooch")) + os.sep + "wsi",
)
mask_path = pooch.retrieve(
    fname="TCGA-AN-A0G0-01Z-00-DX1.mask.png",
    url="https://drive.usercontent.google.com/download?id=17GOOHbL8Bo3933rdIui82akr7stbRfta&export=download&confirm=t",
    known_hash="bb657ead9fd3b8284db6ecc1ca8a1efa57a0e9fd73d2ea63ce6053fbd3d65171",
    path=str(pooch.os_cache("pooch")) + os.sep + "wsi",
)

## Create an encoder model

`tf_encoder` creates encoder models from the available models in `tensorflow.keras.applications`. The input size and type and output pooling are selectable and follow the keras interface. Selecting a dtype of `tensorflow.uint8` allows the model to accept unsigned 8-bit integer inputs instead of float inputs to minimize host to device data transfer. The encoder model is saved into the designated triton server model respository that is mounted within the triton server container.

In [13]:
import numpy as np
from pprint import pprint
from simple_triton.encoders import tf_encoder
from simple_triton.model import TritonModel

# model parameters
keras_name = "EfficientNetV2S"
model_name = f"{keras_name}.tensorflow"
tile = 224
repository = os.path.join(os.environ["HOME"], "models")

# create the model and capture output dimensionality
if not os.path.exists(os.path.join(repository, model_name)):
    dimension_output = tf_encoder(
        repository,
        keras_name,
        model_name,
        input_shape=(tile, tile, 3),
        dtype=tf.uint8,
        pooling="avg",
    )

## Load the model using `TritonModel`

After creating the model, we load the model into Triton using the `TritonModel` class. This class contains methods for loading, unloading, and checking the status of models. To load the model we create a simple configuration with batch size 64, and allow Triton to generate the remaining configuration fields. By default it will load a single copy of the model on each system GPU, and will often automatically set optimizations like pinned memory.

In [14]:
# triton parameters
url = "localhost:8001"  # url for grpc access to tirton server

# load tensorflow model - set maximum batch size
model = TritonModel(model_name, url)
model.load(config={"maxBatchSize": 64})
assert model.is_loaded()
pprint(model.get_config())

context creation failed: Channel.unary_unary() got an unexpected keyword argument '_registered_method'


Exception ignored in: <function InferenceServerClient.__del__ at 0x7f064a64d090>
Traceback (most recent call last):
  File "/home/lac5440/miniconda3/lib/python3.10/site-packages/tritonclient/grpc/_client.py", line 257, in __del__
    self.close()
  File "/home/lac5440/miniconda3/lib/python3.10/site-packages/tritonclient/grpc/_client.py", line 264, in close
    self.stop_stream()
  File "/home/lac5440/miniconda3/lib/python3.10/site-packages/tritonclient/grpc/_client.py", line 1811, in stop_stream
    if self._stream is not None:
AttributeError: 'InferenceServerClient' object has no attribute '_stream'


UnboundLocalError: local variable 'client' referenced before assignment

## Advanced configuration with `ConfigBuilder`

The `ConfigBuilder` class provides access to advanced configuration options like backend optimizations. Here, we create a duplicate model on each GPU (`count=2`) and convert the model to mixed precision to improve speed and memory usage. The model is reloaded using this advanced configuration.

In [ ]:
from simple_triton.config import ConfigBuilder

# initialize builder with a basic configuration
builder = ConfigBuilder(model_name, config={"maxBatchSize": 64})

# increase the number of model instances per GPU to 2
builder.add_instance_group(count=2)

# add automatic mixed precision
builder.add_mixed_precision()

# re-load model with new config
model.load(config=builder.config)

# print config
pprint(model.get_config())

## Run the inference

First, a histomics stream study is created defining the tiles that need to be read based on the whole-slide image, tissue mask, and desired magnification, tile size, and tile overlap. The chunk parameter is used to group tiles during disk reads to maximize throughput. This study initializes a `LargeimagePrefetch` iterator that generates batches of tiles and tile metadata using prefetching.

This iterator is passed to the inference function that is parameterized by the number of tiles per batch, the number of workers, and the maximum number of pending inferences per worker.

In [ ]:
from simple_triton.feature_extraction import inference, study
from simple_triton.tile_iterators import TiffPrefetch
from simple_triton.utils import analyze
from time import time

# slide parameters
batch = 64
magnification = 20.0
chunk = 896
mask_threshold = 0.5

# create a histomics-stream study from a wsi/mask pair
hs_study = study(
    (wsi_path, mask_path),
    t=(tile, tile),
    chunk=(chunk, chunk),
    objective=magnification,
    mask_threshold=mask_threshold,
)

# tile iterator parameters
batch = 64
prefetch = 4
workers = 16  # total number of tile
icc = True  # apply ICC color correction

# inference parameters
limit = 1  # limit on number of pending requests per worker
verbose = True  # display inference statistics and debugging information

# start timer
start = time()

# create tile iterator
iterator = TiffPrefetch(hs_study, np.uint8, icc, batch, prefetch, workers)

# inference
features, metadata, times, failures = inference(
    iterator, model_name, url="localhost:8001", limit=limit, rest=0.0
)

# display elapsed time
print(f"Total elapsed time: {time()-start}")

# analyze performance
analyze(times)

## Write features to .tfr

In [ ]:
from simple_triton.io.tfr_reader import read_record, peek
from simple_triton.io.tfr_writer import write_record

# concatenate features
features = np.concatenate(features[0], axis=0)

# create dummy labels
labels = {"labels": np.random.uniform(size=(10))}

# write to tfrecord
write_record(
    "./triton.tfr", features, metadata, labels, structured=False, precision=tf.float16
)

# get list of .tfr variables for de-serialization
serialized = list(tf.data.TFRecordDataset(["./triton.tfr"]))[0]
variables = peek(serialized)

# verify reading
read_record(serialized, variables, structured=False, precision=tf.float16)